# Samatrix Practice: Student Placement Prediction

Build a classification model that predicts the probability of a student being placed.

Run the notebook from top to bottom. It will create `predictions.csv` with two columns:

- `id`
- `prediction`

For this challenge, `prediction` should be a probability between 0 and 1.

When you reach Step 4, use the **Colab Submission Token** from the Samatrix challenge page. Do not paste your Samatrix username or password in Colab.


## Step 1 : Download the challenge files

Run the setup cell below once. It connects to Samatrix Practice and downloads:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

You do not need to edit this cell.


In [ ]:
#@title Step 1 : Download challenge files { display-mode: "form" }
#@markdown Run this cell once. You do not need to edit the code.

import json
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://practice.samatrix.io"
CHALLENGE_SLUG = "student-placement-001"
FORCE_DOWNLOAD = False

DATA_DIR = Path(".")
CONFIG_URL = f"{BASE_URL}/api/v1/challenges/{CHALLENGE_SLUG}/client-config"

print("Connecting to Samatrix Practice...")
response = requests.get(CONFIG_URL, timeout=30)

if response.status_code != 200:
    raise RuntimeError(f"Could not load challenge config: {response.status_code} {response.text[:300]}")

config = response.json()

files_to_download = {
    "train": "train.csv",
    "test": "test.csv",
    "sample_submission": "sample_submission.csv",
}

for asset_key, filename in files_to_download.items():
    path = DATA_DIR / filename

    if path.exists() and not FORCE_DOWNLOAD:
        print(f"Using existing {filename}")
        continue

    file_url = config["assets"][asset_key]
    file_response = requests.get(file_url, timeout=60)

    if file_response.status_code != 200:
        raise RuntimeError(f"Could not download {filename}: {file_response.status_code}")

    path.write_bytes(file_response.content)
    print(f"Downloaded {filename}")

print()
print("Challenge:", config.get("title"))
print("Metric:", config.get("task", {}).get("metric_name"))
print("Files are ready.")


## 2. Load the data


In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

train.head()


## 3. Train a baseline model

This is a starter baseline. Improve it using better validation, feature engineering, and model selection.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

task_config = config.get("task", {})
target_column = task_config.get("target_column") or "placed"
id_column = task_config.get("id_column") or "id"
prediction_column = task_config.get("prediction_column") or "prediction"

if target_column not in train.columns:
    raise RuntimeError(f"Target column not found in train.csv: {target_column}")

if id_column not in train.columns or id_column not in test.columns:
    raise RuntimeError(f"ID column must exist in both train.csv and test.csv: {id_column}")

X = train.drop(columns=[target_column])
y = train[target_column]
X_test = test.copy()

feature_columns = [column for column in X.columns if column != id_column]

X_encoded = pd.get_dummies(X[feature_columns], dummy_na=True)
X_test_encoded = pd.get_dummies(X_test[feature_columns], dummy_na=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
)

model.fit(X_encoded, y)

test_probabilities = model.predict_proba(X_test_encoded)[:, 1]

submission = pd.DataFrame({
    id_column: test[id_column],
    prediction_column: test_probabilities,
})

submission.to_csv("predictions.csv", index=False)

print("Created predictions.csv")
submission.head()


## 4. Submit to Samatrix Practice

Before running the submit cell:

1. Go back to the Samatrix challenge page.
2. Click **Generate Colab Submission Token**.
3. Copy the **Colab Submission Token**.
4. Run the Step 4 cell in Colab.
5. Paste the token into the form field.
6. Open the **Direct Review Room** link to return to Samatrix Practice and review your result.

Do not paste your Samatrix username or password here. Only the Colab Submission Token is needed.

Colab may show a small **Show code** link for form cells. You can ignore it during normal use.


In [ ]:
#@title Step 4: Run Cell, Paste Token, and Submit { display-mode: "form" }

from pathlib import Path

import requests

EXPECTED_SUBMISSION_ENDPOINT_PATH = "/api/v1/submissions/attempt-token-submit"

submission_config = config.get("submission", {})
endpoint_path = submission_config.get("endpoint_path")

if endpoint_path != EXPECTED_SUBMISSION_ENDPOINT_PATH:
    raise RuntimeError(
        "Samatrix client-config returned an unexpected submission endpoint. "
        f"Expected {EXPECTED_SUBMISSION_ENDPOINT_PATH}, got {endpoint_path!r}."
    )

SUBMISSION_BASE_URL = config.get("base_url") or BASE_URL
SUBMISSION_URL = f"{SUBMISSION_BASE_URL}{endpoint_path}"


def submit_to_samatrix(colab_submission_token: str, predictions_path: str = "predictions.csv") -> dict | None:
    token = (colab_submission_token or "").strip()
    path = Path(predictions_path)

    if not token:
        print("❌ Colab Submission Token is required. Generate a token on the Samatrix challenge page and paste it here.")
        return None

    if not path.exists():
        print("❌ predictions.csv was not found. Run the model cell first.")
        return None

    try:
        with path.open("rb") as file_handle:
            response = requests.post(
                SUBMISSION_URL,
                headers={"Authorization": f"Bearer {token}"},
                files={"predictions_file": ("predictions.csv", file_handle, "text/csv")},
                timeout=60,
            )
    except requests.RequestException as exc:
        print("❌ Could not reach Samatrix Practice. Check your connection and try again.")
        print(str(exc))
        return None

    try:
        data = response.json()
    except ValueError:
        print(f"❌ Samatrix returned a non-JSON response with status {response.status_code}.")
        print(response.text[:1000])
        return None

    if response.status_code != 200:
        print(f"❌ Submission failed with status {response.status_code}.")
        print(data.get("detail") or json.dumps(data, indent=2))
        return data

    evaluation = data.get("evaluation") or {}

    print("✅ Submission completed.")
    print("Score:", evaluation.get("score_normalized"))
    print("Metric:", evaluation.get("metric_name"))
    print("Metric value:", evaluation.get("metric_value"))
    print("Review Room:", data.get("review_url") or data.get("review_full_url"))

    return data


try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    token_box = widgets.Password(
        description="Colab Submission Token:",
        placeholder="Paste token from Samatrix",
        layout=widgets.Layout(width="95%"),
        style={"description_width": "120px"},
    )

    submit_button = widgets.Button(
        description="Open Direct Review Room",
        button_style="success",
        tooltip="Paste token and submit",
    )

    output = widgets.Output()

    def _on_submit_clicked(_):
        with output:
            clear_output()
            submit_to_samatrix(token_box.value)

    submit_button.on_click(_on_submit_clicked)

    display(token_box)
    display(submit_button)
    display(output)

except Exception as exc:
    print("Interactive button could not be loaded in this notebook environment.")
    print("Use this fallback instead:")
    print("from getpass import getpass")
    print("submit_to_samatrix(getpass('Paste Colab Submission Token: '))")
    print("Widget error:", exc)
